# 🚀 SpaceX Falcon 9 – Notebook 6: Interactive Map with Folium

In [ ]:
!pip install folium --quiet
import folium
import pandas as pd
import numpy as np
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

df = pd.read_csv('dataset_part_2.csv')
print("Ready!")

## Launch Sites Map

In [ ]:
# Base map centered on USA
site_map = folium.Map(location=[29.559684888503615, -95.0830971930759], zoom_start=5)

launch_sites = df.groupby('LaunchSite').agg({'Latitude':'first','Longitude':'first'}).reset_index()

colors = ['red','blue','green','purple','orange']
for i, row in launch_sites.iterrows():
    folium.Circle(
        [row['Latitude'], row['Longitude']],
        radius=1000, color=colors[i % len(colors)], fill=True
    ).add_to(site_map)
    folium.Marker(
        [row['Latitude'], row['Longitude']],
        icon=DivIcon(icon_size=(250,36), icon_anchor=(0,0),
                     html=f'<div style="font-size:12px;color:black;font-weight:bold;">{row["LaunchSite"]}</div>')
    ).add_to(site_map)

site_map.save('launch_sites_map.html')
site_map

## Success/Failure Markers per Launch

In [ ]:
marker_map = folium.Map(location=[29.559684888503615, -95.0830971930759], zoom_start=5)
marker_cluster = MarkerCluster().add_to(marker_map)

for i, row in df.iterrows():
    color = 'green' if row['Class'] == 1 else 'red'
    icon_type = 'ok-sign' if row['Class'] == 1 else 'remove-sign'
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Site: {row['LaunchSite']}\nOutcome: {row['Outcome']}\nPayload: {row['PayloadMass']} kg",
        icon=folium.Icon(color=color, icon=icon_type, prefix='glyphicon')
    ).add_to(marker_cluster)

marker_map.save('launch_outcomes_map.html')
marker_map

## Distance to Coastline / City (proximity analysis)

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Example: distance from KSC LC-39A to nearest city (Titusville, FL)
d = haversine_distance(28.573, -80.647, 28.612, -80.807)
print(f"KSC LC-39A to Titusville: {d:.2f} km")